# Notebook 04: Gold Facts
## Proposito
Construir las 2 tablas de hechos del modelo, con lookups SCD2-aware:
- fact_order_items (grain: item de orden)
- fact_reviews (grain: review)
## Cross-lakehouse pattern
- Reads silver: spark.read.table(f"{SILVER_LAKEHOUSE}.{table}")
- Reads gold: spark.read.table(f"{GOLD_LAKEHOUSE}.{table}")
- Writes gold: df.write.saveAsTable(f"{GOLD_LAKEHOUSE}.{table}")
## Punto tecnico clave
El lookup de customer_sk usa rango temporal:

fact.timestamp >= dim_customer.valid_from AND fact.timestamp < dim_customer.valid_to

Esto asegura que cada hecho referencia la version del cliente vigente al momento del evento (no al actual)

### Ejemplo de lookup temporal (SCD Type 2)

Supongamos que el cliente **C001** vivía en **San José** hasta el 31-May-2024 y luego se mudó a **Cartago**.

**Dimensión de clientes (`dim_customer`)**

| customer_sk | customer_id | ciudad   | valid_from | valid_to   |
|-------------|-------------|----------|------------|------------|
| 101 | C001 | San José | 2024-01-01 | 2024-06-01 |
| 102 | C001 | Cartago  | 2024-06-01 | 9999-12-31 |

**Hechos (`fact_sales`)**

| sale_id | customer_id | timestamp |
|----------|-------------|------------|
| 1 | C001 | 2024-03-15 |
| 2 | C001 | 2024-08-10 |

El lookup del `customer_sk` se realiza usando:

```sql
fact.timestamp >= dim_customer.valid_from
AND fact.timestamp < dim_customer.valid_to

Cada hecho queda asociado a la versión del cliente que estaba vigente cuando ocurrió el evento


In [25]:
# **Celda 2 — PARAMETERS CELL:**

SILVER_LAKEHOUSE = "lh_olist_silver"
GOLD_LAKEHOUSE = "lh_olist_gold"
WRITE_MODE = "overwrite"

StatementMeta(, 8c255173-d597-4879-9608-55654979f88f, 27, Finished, Available, Finished, False)

In [26]:
# **Celda 3 — Imports y helpers:**

from pyspark.sql.functions import(
    col, lit, coalesce, when,
    date_format, current_timestamp
)

from pyspark.sql.types import IntegerType

def read_silver(t): return spark.read.table(f"{SILVER_LAKEHOUSE}.{t}")
def read_gold(t):   return spark.read.table(f"{GOLD_LAKEHOUSE}.{t}")
def write_gold(df, t):
    (df.write.format("delta")
       .mode(WRITE_MODE)
       .option("overwriteSchema", "true")
       .saveAsTable(f"{GOLD_LAKEHOUSE}.{t}"))

StatementMeta(, 8c255173-d597-4879-9608-55654979f88f, 28, Finished, Available, Finished, False)

In [27]:
# **Celda 4 — Cargar todas las dimensiones y silvers necesarios:**

silver_order_items = read_silver("silver_order_items")
silver_orders = read_silver("silver_orders")
silver_customers = read_silver("silver_customers")
silver_payments = read_silver("silver_order_payments")
silver_reviews = read_silver("silver_order_reviews")

# Gold dims
dim_customer = read_gold("dim_customer")
dim_product = read_gold("dim_product")
dim_seller = read_gold("dim_seller")
dim_order_junk = read_gold("dim_order_junk")

print("Tablas cargadas en memoria")

StatementMeta(, 8c255173-d597-4879-9608-55654979f88f, 29, Finished, Available, Finished, False)

Tablas cargadas en memoria


In [9]:
# **Celda 5 — fact_order_items: paso 1 (join base):**

print("fact_order_items — Paso 1: join base")

items_with_order = (silver_order_items
    .join(silver_orders.select(
        "order_id", "customer_id",
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_status", "delivery_status_bucket"),
        on = "order_id", how = "inner"))

items_with_unique = (items_with_order
    .join(silver_customers.select("customer_id", "customer_unique_id"),
            on = "customer_id", how = "inner"))

print(f"Filas: {items_with_unique.count():,}")

StatementMeta(, 8c255173-d597-4879-9608-55654979f88f, 11, Finished, Available, Finished, False)

fact_order_items — Paso 1: join base
Filas: 112,650


In [10]:
# **Celda 6 — Paso 2: SCD2 lookup de customer_sk:**

items_with_customer_sk = (items_with_unique.alias("i")
    .join(dim_customer.alias("dc"),
            (col("i.customer_unique_id") == col("dc.customer_natural_id")) &
            (col("i.order_purchase_timestamp") >= col("dc.valid_from")) &
            (col("i.order_purchase_timestamp") <  col("dc.valid_to")),
            "left")
    .select("i.*", col("dc.customer_sk")))

con_sk = items_with_customer_sk.filter(col("customer_sk").isNotNull()).count()
sin_sk = items_with_customer_sk.filter(col("customer_sk").isNull()).count()
total = items_with_customer_sk.count()

print(f"Items con customer_sk asignado: {con_sk:,} ({100*con_sk/total:.2f}%)")
print(f"Items sin match (revisar):{sin_sk:,} ({100*sin_sk/total:.2f}%)")

if sin_sk > total * 0.01:

    print(f"Mas del 1% sin match — revisar la lógica de SCD2")

#**Si el % sin match es alto revisar

StatementMeta(, 8c255173-d597-4879-9608-55654979f88f, 12, Finished, Available, Finished, False)

Items con customer_sk asignado: 112,650 (100.00%)
Items sin match (revisar):0 (0.00%)


In [18]:
# **Celda 7 — Paso 3: lookups SCD1 (product, seller):**

print("Paso 3: lookups product_sk, seller_sk")

items_with_product = (items_with_customer_sk.alias("i")
    .join(dim_product.select("product_sk",
                            #selecciona el id, le cambia el nombre a la col y luego renombra la tabla dp
                             col("product_natural_id").alias("_pnid")).alias("dp"),
          col("i.product_id") == col("dp._pnid"),
          "left")
    #elimina la columna 
    .drop("_pnid"))

items_with_seller = (items_with_product.alias("i")
    .join(dim_seller.select("seller_sk",
                            col("seller_natural_id").alias("_snid")).alias("ds"),
          col("i.seller_id") == col("ds._snid"),
          "left")
    .drop("_snid"))

StatementMeta(, 8c255173-d597-4879-9608-55654979f88f, 20, Finished, Available, Finished, False)

Paso 3: lookups product_sk, seller_sk


In [20]:
#**Celda 8 — Paso 4: lookup junk_sk:**

print("Paso 4: lookup junk_sk")

payment_main = (silver_payments
    .filter(col("payment_sequential") == 1)
    .select("order_id", "payment_type", "installments_bucket"))

review_flag = (silver_reviews
    .select("order_id").distinct()
    .withColumn("has_review", lit(True)))

items_with_junk_attrs = (items_with_seller
    .join(payment_main, on="order_id", how="left")
    .join(review_flag,  on="order_id", how="left")
    .withColumn("has_review",coalesce(col("has_review"),lit(False)))
    .withColumn("payment_type", coalesce(col("payment_type"), lit("none")))
    .withColumn("installments_bucket", coalesce(col("installments_bucket"), lit("none"))))

items_with_junk = (items_with_junk_attrs.alias("i")
    .join(dim_order_junk.alias("dj"),
        (col("i.order_status")== col("dj.order_status")) &
        (col("i.delivery_status_bucket") == col("dj.delivery_status_bucket")) &
        (col("i.payment_type") == col("dj.payment_type")) &
        (col("i.installments_bucket") == col("dj.installments_bucket")) &
        (col("i.has_review") == col("dj.has_review")),
        "left")
    .select("i.*", col("dj.junk_sk")))


StatementMeta(, 8c255173-d597-4879-9608-55654979f88f, 22, Finished, Available, Finished, False)

Paso 4: lookup junk_sk


In [22]:
#**Celda 9 — Paso 5: date keys y select final:**

print("Paso 5: date keys y persistencia")

fact_order_items = (items_with_junk
    .withColumn("date_sk_purchase",
        date_format(col("order_purchase_timestamp"), "yyyyMMdd").cast(IntegerType()))
    .withColumn("date_sk_delivered",
        date_format(col("order_delivered_customer_date"), "yyyyMMdd").cast(IntegerType()))
    .select(
        #degenerate dimensions
        "order_id",
        "order_item_id",
        #FKs to dimensions
        "customer_sk", #SCD2-aware
        "product_sk",
        "seller_sk",
        "junk_sk",
        "date_sk_purchase",
        "date_sk_delivered",
        #measures
        "price",
        "freight_value",
        "total_item_value"

    ))

write_gold(fact_order_items, "fact_order_items")

StatementMeta(, 8c255173-d597-4879-9608-55654979f88f, 24, Finished, Available, Finished, False)

Paso 5: date keys y persistencia


In [28]:
# **Celda 10 — fact_reviews:**

print("fact_reviews")

reviews_with_customer = (silver_reviews
    .join(silver_orders.select("order_id", "customer_id"), on = "order_id", how = "left")
    .join(silver_customers.select("customer_id", "customer_unique_id"), on = "customer_id", how = "left"))

#SCD2 lookup respecto a review_creation_date
reviews_with_sk = (reviews_with_customer.alias("r")
    .join(dim_customer.alias("dc"),
        (col("r.customer_unique_id") == col("dc.customer_natural_id")) &
        (col("r.review_creation_date") >= col("dc.valid_from")) &
        (col("r.review_creation_date") <  col("dc.valid_to")),
        "left")
    .select("r.*", col("dc.customer_sk")))

fact_reviews = (reviews_with_sk
    .withColumn("date_sk_review",
        date_format(col("review_creation_date"), "yyyyMMdd").cast(IntegerType()))
    .select(
        #degenerate dimensions
        "review_id",
        "order_id",
        #FKs CONFORMED (mismas qie fact_order_items)
        "customer_sk",
        "date_sk_review",
        #measure
        "review_score",
        "response_time_hours",
        "has_comment",
        "comment_length"

    ))

write_gold(fact_reviews, "fact_reviews")

StatementMeta(, 8c255173-d597-4879-9608-55654979f88f, 30, Finished, Available, Finished, False)

fact_reviews


# Nota: ~30 reviews (0.03%) tienen review_creation_date anterior al order_purchase_timestamp
# de la orden asociada. Esto causa que el lookup SCD2 falle (review cae fuera de [valid_from, valid_to)).
# Es ruido conocido del dataset Olist y se acepta bajo el umbral del 0.1%.

In [38]:
print("=" * 60)
print("VALIDACION INTEGRIDAD REFERENCIAL")
print("=" * 60)

# Umbral tolerable de huérfanos
THRESHOLD_PCT = 0.1  # 0.1% — ruido aceptable de calidad de datos

f = read_gold("fact_order_items")
total_items = f.count()
print(f"\nact_order_items: {total_items:,} filas\n")

INTEGRIDAD_ITEMS = [
    ("customer_sk","dim_customer","customer_sk"),
    ("product_sk","dim_product","product_sk"),
    ("seller_sk","dim_seller","seller_sk"),
    ("junk_sk","dim_order_junk","junk_sk"),
    ("date_sk_purchase", "dim_date","date_sk"),
]

problemas = 0
for fk, dim, dim_pk in INTEGRIDAD_ITEMS:
    huerfanos = (f.alias("f")
        .join(read_gold(dim).alias("d"), col(f"f.{fk}") == col(f"d.{dim_pk}"), "left_anti")
        .count())
    pct = (huerfanos / total_items) * 100
    if huerfanos == 0:
        status = "Correcto"
    elif pct <= THRESHOLD_PCT:
        status = "Bajo umbral"
    else:
        status = "Supera umbral"
        problemas += 1
    print(f"{status} {fk:10} -- {dim:10}: {huerfanos:,} huérfanos ({pct:.3f}%)")

fr = read_gold("fact_reviews")
total_reviews = fr.count()
print(f"\nfact_reviews: {total_reviews:,} filas\n")

for fk, dim, dim_pk in [
    ("customer_sk","dim_customer","customer_sk"),
    ("date_sk_review","dim_date","date_sk"),
]:
    huerfanos = (fr.alias("f")
        .join(read_gold(dim).alias("d"), col(f"f.{fk}") == col(f"d.{dim_pk}"), "left_anti")
        .count())
    pct = (huerfanos / total_reviews) * 100
    if huerfanos == 0:
        status = "Correcto"
    elif pct <= THRESHOLD_PCT:
        status = "Bajo umbral"
    else:
        status = "Supera umbral"
        problemas += 1
    print(f"{status} {fk:10} -- {dim:10}: {huerfanos:,} huérfanos ({pct:.3f}%)")

if problemas:
    raise Exception(f"{problemas} relación(es) supera(n) el umbral de {THRESHOLD_PCT}%")
print(f"\nIntegridad referencial dentro de umbrales tolerables")

StatementMeta(, 8c255173-d597-4879-9608-55654979f88f, 40, Finished, Available, Finished, False)

VALIDACIÓN INTEGRIDAD REFERENCIAL

act_order_items: 112,650 filas

  Correcto customer_sk -- dim_customer: 0 huérfanos (0.000%)
  Correcto product_sk -- dim_product: 0 huérfanos (0.000%)
  Correcto seller_sk  -- dim_seller: 0 huérfanos (0.000%)
  Correcto junk_sk    -- dim_order_junk: 0 huérfanos (0.000%)
  Correcto date_sk_purchase -- dim_date  : 0 huérfanos (0.000%)

fact_reviews: 98,410 filas

  Bajo umbral customer_sk -- dim_customer: 30 huérfanos (0.030%)
  Correcto date_sk_review -- dim_date  : 0 huérfanos (0.000%)

Integridad referencial dentro de umbrales tolerables


In [35]:
silver_reviews_full = spark.read.table(f"{SILVER_LAKEHOUSE}.silver_order_reviews")

# Reconstruir los huérfanos con todos los timestamps relevantes
huerfanos_detalle = (huerfanos.select("review_id", "order_id")
    .join(silver_orders.select("order_id", "customer_id", "order_purchase_timestamp"),
          on="order_id", how="left")
    .join(silver_customers.select("customer_id", "customer_unique_id"),
          on="customer_id", how="left")
    .join(silver_reviews_full.select("review_id", "review_creation_date"),
          on="review_id", how="left"))

# Para cada huérfano, traer el valid_from MÍNIMO del cliente en dim_customer
from pyspark.sql.functions import min as spark_min

first_valid_from = (dc.groupBy("customer_natural_id")
    .agg(spark_min("valid_from").alias("earliest_valid_from")))

diagnostico = (huerfanos_detalle.alias("h")
    .join(first_valid_from.alias("fvf"),
          col("h.customer_unique_id") == col("fvf.customer_natural_id"),
          "left")
    .select("review_id",
            "order_purchase_timestamp",
            "review_creation_date",
            "earliest_valid_from"))

diagnostico.show(30, truncate=False)

StatementMeta(, 8c255173-d597-4879-9608-55654979f88f, 37, Finished, Available, Finished, False)

+--------------------------------+------------------------+--------------------+-------------------+
|review_id                       |order_purchase_timestamp|review_creation_date|earliest_valid_from|
+--------------------------------+------------------------+--------------------+-------------------+
|0f54fea9e89c2a9398b2bd56e3880eda|2018-08-29 16:27:49     |2018-08-26 00:00:00 |2018-08-29 16:27:49|
|68c3385b4bb41af5847346c3552ba744|2018-08-07 11:16:28     |2018-08-04 00:00:00 |2018-08-07 11:16:28|
|6c20e65e7fcad5e437f5111232bd450b|2018-08-10 18:54:48     |2018-05-05 00:00:00 |2018-08-10 18:54:48|
|51962867e91b52724f73c8e445a69074|2018-08-22 18:52:29     |2018-08-21 00:00:00 |2018-08-22 18:52:29|
|7e3428c8014fc4a420ec67a216b78c69|2018-08-20 14:17:09     |2018-07-25 00:00:00 |2018-08-20 14:17:09|
|9c0d840dfe562debfa52a1792bd64bff|2018-08-28 15:26:39     |2018-08-28 00:00:00 |2018-08-28 15:26:39|
|18071cb247f057c348988037fa94a0cd|2018-08-09 14:54:47     |2018-08-02 00:00:00 |2018-08-09 